# NanoVision Net X — Image Reconstruction (Colab)

This notebook adds an **image reconstruction workflow** for NanoVision Net X using a convolutional autoencoder in TensorFlow/Keras.

It is designed for Google Colab and focuses on:
- training a reconstruction model on microscopy-like grayscale images
- improving image quality from noisy/low-quality inputs
- reporting reconstruction quality with **PSNR** and **SSIM**
- visualizing input vs reconstructed outputs side by side


In [ ]:
# Install dependencies (Colab-safe)
!pip -q install scikit-image tensorflow


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


In [ ]:
# Data: use MNIST as a stand-in grayscale microscopy-like dataset for reconstruction experiments
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Add channel dimension
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# Add synthetic noise to simulate low-quality microscopy captures
def add_noise(x, noise_factor=0.35):
    noisy = x + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x.shape)
    return np.clip(noisy, 0.0, 1.0)

x_train_noisy = add_noise(x_train)
x_test_noisy = add_noise(x_test)

print("Train shape:", x_train.shape, "Noisy train shape:", x_train_noisy.shape)
print("Test shape:", x_test.shape, "Noisy test shape:", x_test_noisy.shape)


In [ ]:
# NanoVision Net X style reconstruction model (convolutional autoencoder)

def build_autoencoder(input_shape=(28, 28, 1)):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2), padding="same")(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    encoded = layers.MaxPooling2D((2, 2), padding="same")(x)

    # Decoder
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(encoded)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(x)
    x = layers.UpSampling2D((2, 2))(x)
    outputs = layers.Conv2D(1, (3, 3), activation="sigmoid", padding="same")(x)

    model = models.Model(inputs, outputs, name="NanoVisionNetX_Reconstructor")
    model.compile(optimizer="adam", loss="mse")
    return model

model = build_autoencoder()
model.summary()


In [ ]:
# Train reconstruction model
EPOCHS = 5
BATCH_SIZE = 128

history = model.fit(
    x_train_noisy,
    x_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


In [ ]:
# Reconstruct noisy test images
reconstructed = model.predict(x_test_noisy, verbose=0)

# Compute quality metrics on a subset for speed
subset = 1000
psnr_scores = []
ssim_scores = []

for i in range(subset):
    gt = x_test[i, ..., 0]
    rec = reconstructed[i, ..., 0]
    psnr_scores.append(psnr(gt, rec, data_range=1.0))
    ssim_scores.append(ssim(gt, rec, data_range=1.0))

print(f"Average PSNR on {subset} samples: {np.mean(psnr_scores):.2f} dB")
print(f"Average SSIM on {subset} samples: {np.mean(ssim_scores):.4f}")


In [ ]:
# Visualize reconstruction quality
num_images = 8
idx = np.random.choice(len(x_test), size=num_images, replace=False)

plt.figure(figsize=(16, 6))
for i, j in enumerate(idx):
    # Noisy input
    ax = plt.subplot(3, num_images, i + 1)
    plt.imshow(x_test_noisy[j].squeeze(), cmap="gray")
    plt.title("Noisy")
    plt.axis("off")

    # Reconstructed
    ax = plt.subplot(3, num_images, i + 1 + num_images)
    plt.imshow(reconstructed[j].squeeze(), cmap="gray")
    plt.title("Reconstructed")
    plt.axis("off")

    # Ground truth
    ax = plt.subplot(3, num_images, i + 1 + 2 * num_images)
    plt.imshow(x_test[j].squeeze(), cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

plt.suptitle("NanoVision Net X Image Reconstruction Results", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Optional: save model and a sample output image
os.makedirs("reconstruction_outputs", exist_ok=True)
model.save("reconstruction_outputs/nanovision_reconstructor.keras")

sample_idx = 0
plt.imsave(
    "reconstruction_outputs/sample_reconstruction.png",
    reconstructed[sample_idx].squeeze(),
    cmap="gray"
)

print("Saved:")
print("- reconstruction_outputs/nanovision_reconstructor.keras")
print("- reconstruction_outputs/sample_reconstruction.png")
